In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/denkuznetz/food-delivery-time-prediction/Food_Delivery_Times.csv


In [8]:
"""
Food Delivery Time Prediction - Regression Model Comparison
============================================================
Models: Linear, Polynomial, Ridge, Lasso, ElasticNet, Polynomial+regularised,
plus Random Forest and Gradient Boosting as non-linear baselines.

Dataset columns (Food Delivery Time Prediction):
    Order_ID, Distance_km, Weather, Traffic_Level, Time_of_Day, Vehicle_Type,
    Preparation_Time_min, Courier_Experience_yrs, Delivery_Time_min (target)

Usage:
    python delivery_time_regression.py --csv Food_Delivery_Times.csv   # real data
    python delivery_time_regression.py                                  # synthetic demo
"""
import argparse
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

TARGET = "Delivery_Time_min"
ID_COLS = ["Order_ID"]


# ----------------------------------------------------------------------------
# 1. Data
# ----------------------------------------------------------------------------
def make_synthetic_data(n=1000, seed=RANDOM_STATE):
    """Synthetic data mimicking the real dataset's schema and value ranges."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "Order_ID": np.arange(n),
        "Distance_km": rng.uniform(0.6, 20, n).round(2),
        "Weather": rng.choice(["Clear", "Rainy", "Snowy", "Foggy", "Windy"], n,
                              p=[.45, .2, .1, .1, .15]),
        "Traffic_Level": rng.choice(["Low", "Medium", "High"], n, p=[.3, .4, .3]),
        "Time_of_Day": rng.choice(["Morning", "Afternoon", "Evening", "Night"], n),
        "Vehicle_Type": rng.choice(["Bike", "Scooter", "Car"], n, p=[.5, .3, .2]),
        "Preparation_Time_min": rng.integers(5, 30, n),
        "Courier_Experience_yrs": rng.integers(0, 10, n).astype(float),
    })
    weather_pen = df.Weather.map({"Clear": 0, "Windy": 1.5, "Foggy": 3, "Rainy": 5, "Snowy": 8})
    traffic_pen = df.Traffic_Level.map({"Low": 0, "Medium": 4, "High": 9})
    tod_pen = df.Time_of_Day.map({"Morning": 0, "Afternoon": 1, "Evening": 3, "Night": 1.5})
    veh_factor = df.Vehicle_Type.map({"Bike": 3.0, "Scooter": 2.6, "Car": 2.9})
    df[TARGET] = (
        10 + 0.95 * df.Preparation_Time_min
        + veh_factor * df.Distance_km + 0.05 * df.Distance_km ** 2     # mildly non-linear
        + weather_pen + traffic_pen + tod_pen
        - 0.6 * df.Courier_Experience_yrs
        + rng.normal(0, 4, n)
    ).round().clip(8, None)
    # real data has a few missing values - exercise the imputers
    for col in ["Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs"]:
        df.loc[rng.random(n) < 0.03, col] = np.nan
    return df


KAGGLE_CSV = ("/kaggle/input/datasets/denkuznetz/food-delivery-time-prediction/"
              "Food_Delivery_Times.csv")


def load_data(csv_path):
    import os
    if not csv_path and os.path.exists(KAGGLE_CSV):   # running inside the Kaggle notebook
        csv_path = KAGGLE_CSV
    if csv_path:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=[TARGET])
        print(f"Loaded {csv_path}")
    else:
        df = make_synthetic_data()
        print("No --csv given: using synthetic data with the same schema")
    return df.drop(columns=[c for c in ID_COLS if c in df.columns])


# ----------------------------------------------------------------------------
# 2. Preprocessing
# ----------------------------------------------------------------------------
def build_preprocessor(X, scale=True):
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = X.select_dtypes(exclude="number").columns.tolist()
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ])


# ----------------------------------------------------------------------------
# 3. Models + hyper-parameter grids
# ----------------------------------------------------------------------------
def get_models(X):
    def pipe(est, poly_degree=None, scale=True):
        steps = [("prep", build_preprocessor(X, scale))]
        if poly_degree:
            steps.append(("poly", PolynomialFeatures(poly_degree, include_bias=False)))
            steps.append(("sc2", StandardScaler()))   # re-scale after expansion
        steps.append(("model", est))
        return Pipeline(steps)

    return {
        "Linear": (pipe(LinearRegression()), {}),
        "Polynomial (deg 2)": (pipe(LinearRegression(), poly_degree=2), {}),
        "Polynomial (deg 3) + Ridge": (
            pipe(Ridge(), poly_degree=3), {"model__alpha": [10, 100, 1000, 3000]}),
        "Ridge": (pipe(Ridge()), {"model__alpha": np.logspace(-3, 2, 6)}),
        "Lasso": (pipe(Lasso(max_iter=20000)), {"model__alpha": np.logspace(-4, 0, 6)}),
        "ElasticNet": (
            pipe(ElasticNet(max_iter=20000)),
            {"model__alpha": np.logspace(-4, 0, 5), "model__l1_ratio": [0.1, 0.5, 0.9]}),
        "Poly(2) + ElasticNet": (
            pipe(ElasticNet(max_iter=20000), poly_degree=2),
            {"model__alpha": [0.001, 0.01, 0.1], "model__l1_ratio": [0.2, 0.5, 0.8]}),
        "Random Forest": (
            pipe(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), scale=False),
            {"model__n_estimators": [200], "model__max_depth": [None, 8],
             "model__min_samples_leaf": [1, 5]}),
        "Gradient Boosting": (
            pipe(GradientBoostingRegressor(random_state=RANDOM_STATE), scale=False),
            {"model__n_estimators": [100, 300], "model__learning_rate": [0.03, 0.1],
             "model__max_depth": [2, 3]}),
    }


# ----------------------------------------------------------------------------
# 4. Train / evaluate
# ----------------------------------------------------------------------------
def evaluate(name, model, X_tr, y_tr, X_te, y_te, cv):
    cv_rmse = -cross_val_score(model, X_tr, y_tr, cv=cv,
                               scoring="neg_root_mean_squared_error").mean()
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return {
        "Model": name,
        "CV RMSE": cv_rmse,
        "Test RMSE": np.sqrt(mean_squared_error(y_te, pred)),
        "Test MAE": mean_absolute_error(y_te, pred),
        "Test R2": r2_score(y_te, pred),
    }, pred


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=None)
    ap.add_argument("--out", default=".")
    args, _ = ap.parse_known_args()   # tolerant of Jupyter/Kaggle's extra argv

    df = load_data(args.csv)
    print(f"Data: {df.shape[0]} rows, {df.shape[1]-1} features, target='{TARGET}'")
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)

    results, preds, fitted = [], {}, {}
    for name, (pipeline, grid) in get_models(X_tr).items():
        if grid:
            search = GridSearchCV(pipeline, grid, cv=cv,
                                  scoring="neg_root_mean_squared_error", n_jobs=-1)
            search.fit(X_tr, y_tr)
            model, best = search.best_estimator_, search.best_params_
        else:
            model, best = pipeline, {}
        row, pred = evaluate(name, model, X_tr, y_tr, X_te, y_te, cv)
        row["Best params"] = {k.replace("model__", ""): (round(float(v), 4) if isinstance(v, (float, np.floating)) else v)
                              for k, v in best.items()}
        results.append(row)
        preds[name], fitted[name] = pred, model
        print(f"{name:30s} Test RMSE={row['Test RMSE']:.3f}  R2={row['Test R2']:.3f}  {row['Best params']}")

    res = pd.DataFrame(results).sort_values("Test RMSE").reset_index(drop=True)
    res.to_csv(f"{args.out}/model_comparison.csv", index=False)
    print("\n=== Leaderboard ===")
    print(res.drop(columns="Best params").round(3).to_string(index=False))

    best_name = res.loc[0, "Model"]
    print(f"\nBest model: {best_name}")

    # --- Plots ---
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].barh(res["Model"][::-1], res["Test RMSE"][::-1], color="#4C78A8")
    ax[0].set_xlabel("Test RMSE (minutes)"); ax[0].set_title("Model comparison (lower is better)")
    ax[1].scatter(y_te, preds[best_name], s=10, alpha=.5)
    lim = [y_te.min(), y_te.max()]
    ax[1].plot(lim, lim, "r--"); ax[1].set_xlabel("Actual (min)"); ax[1].set_ylabel("Predicted (min)")
    ax[1].set_title(f"{best_name}: predicted vs actual")
    plt.tight_layout(); plt.savefig(f"{args.out}/results.png", dpi=130); plt.close()

    # --- Sparsity / coefficients from Lasso & ElasticNet ---
    for nm in ["Lasso", "ElasticNet"]:
        m = fitted[nm]
        names = m.named_steps["prep"].get_feature_names_out()
        coefs = pd.Series(m.named_steps["model"].coef_, index=names)
        print(f"\n{nm}: kept {int((coefs != 0).sum())}/{len(coefs)} features; top effects:")
        print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

    # --- Residual check on best model ---
    resid = y_te - preds[best_name]
    plt.figure(figsize=(6, 4)); plt.hist(resid, bins=30, color="#59A14F")
    plt.title(f"Residuals - {best_name}"); plt.xlabel("minutes")
    plt.tight_layout(); plt.savefig(f"{args.out}/residuals.png", dpi=130); plt.close()

    import joblib
    joblib.dump(fitted[best_name], f"{args.out}/best_model.joblib")
    print("\nSaved: model_comparison.csv, results.png, residuals.png, best_model.joblib")


if __name__ == "__main__":
    main()

Loaded /kaggle/input/datasets/denkuznetz/food-delivery-time-prediction/Food_Delivery_Times.csv
Data: 1000 rows, 7 features, target='Delivery_Time_min'
Linear                         Test RMSE=8.952  R2=0.821  {}
Polynomial (deg 2)             Test RMSE=9.790  R2=0.786  {}
Polynomial (deg 3) + Ridge     Test RMSE=9.602  R2=0.794  {'alpha': 1000}
Ridge                          Test RMSE=8.977  R2=0.820  {'alpha': 10.0}
Lasso                          Test RMSE=8.964  R2=0.821  {'alpha': 0.0251}
ElasticNet                     Test RMSE=9.042  R2=0.818  {'alpha': 0.1, 'l1_ratio': 0.9}
Poly(2) + ElasticNet           Test RMSE=9.508  R2=0.798  {'alpha': 0.1, 'l1_ratio': 0.8}
Random Forest                  Test RMSE=9.652  R2=0.792  {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
Gradient Boosting              Test RMSE=9.542  R2=0.797  {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}

=== Leaderboard ===
                     Model  CV RMSE  Test RMSE  Test MAE  Test

In [10]:
"""
Accuracy evaluation for the food-delivery-time regression models.

Self-contained: paste it into one notebook cell (or save as a .py) and run it. Nothing else needed.

What it does
  1. Repeated 5-fold CV (x2) with NESTED tuning -> honest, leakage-free error estimates
  2. Compares every model against a naive "always predict the mean" baseline
  3. Reports RMSE, MAE, R2, MAPE and % of orders predicted within +/-5 and +/-10 minutes
  4. For the best model: out-of-fold error broken down by distance, traffic, weather, etc.
  5. Diagnostic plots (predicted vs actual, residuals vs predicted)

Usage:
    python evaluate_accuracy.py --csv Food_Delivery_Times.csv
    (inside Kaggle the CSV path is auto-detected)
"""
import argparse
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.model_selection import (GridSearchCV, KFold, RepeatedKFold,
                                     cross_val_predict, cross_validate)

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Data + models (self-contained: no other file needed)
# ----------------------------------------------------------------------------
RANDOM_STATE = 42
TARGET = "Delivery_Time_min"
ID_COLS = ["Order_ID"]
KAGGLE_CSV = ("/kaggle/input/datasets/denkuznetz/food-delivery-time-prediction/"
              "Food_Delivery_Times.csv")


def make_synthetic_data(n=1000, seed=RANDOM_STATE):
    """Synthetic data with the same schema - only used when no CSV is found."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "Order_ID": np.arange(n),
        "Distance_km": rng.uniform(0.6, 20, n).round(2),
        "Weather": rng.choice(["Clear", "Rainy", "Snowy", "Foggy", "Windy"], n,
                              p=[.45, .2, .1, .1, .15]),
        "Traffic_Level": rng.choice(["Low", "Medium", "High"], n, p=[.3, .4, .3]),
        "Time_of_Day": rng.choice(["Morning", "Afternoon", "Evening", "Night"], n),
        "Vehicle_Type": rng.choice(["Bike", "Scooter", "Car"], n, p=[.5, .3, .2]),
        "Preparation_Time_min": rng.integers(5, 30, n),
        "Courier_Experience_yrs": rng.integers(0, 10, n).astype(float),
    })
    weather_pen = df.Weather.map({"Clear": 0, "Windy": 1.5, "Foggy": 3, "Rainy": 5, "Snowy": 8})
    traffic_pen = df.Traffic_Level.map({"Low": 0, "Medium": 4, "High": 9})
    tod_pen = df.Time_of_Day.map({"Morning": 0, "Afternoon": 1, "Evening": 3, "Night": 1.5})
    veh_factor = df.Vehicle_Type.map({"Bike": 3.0, "Scooter": 2.6, "Car": 2.9})
    df[TARGET] = (
        10 + 0.95 * df.Preparation_Time_min
        + veh_factor * df.Distance_km + 0.05 * df.Distance_km ** 2
        + weather_pen + traffic_pen + tod_pen
        - 0.6 * df.Courier_Experience_yrs
        + rng.normal(0, 4, n)
    ).round().clip(8, None)
    for col in ["Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs"]:
        df.loc[rng.random(n) < 0.03, col] = np.nan
    return df


def load_data(csv_path):
    import os
    if not csv_path and os.path.exists(KAGGLE_CSV):   # running inside the Kaggle notebook
        csv_path = KAGGLE_CSV
    if csv_path:
        df = pd.read_csv(csv_path).dropna(subset=[TARGET])
        print(f"Loaded {csv_path}")
    else:
        df = make_synthetic_data()
        print("No CSV found: using synthetic data with the same schema")
    return df.drop(columns=[c for c in ID_COLS if c in df.columns])


def build_preprocessor(X, scale=True):
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = X.select_dtypes(exclude="number").columns.tolist()
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ])


def get_models(X):
    def pipe(est, poly_degree=None, scale=True):
        steps = [("prep", build_preprocessor(X, scale))]
        if poly_degree:
            steps.append(("poly", PolynomialFeatures(poly_degree, include_bias=False)))
            steps.append(("sc2", StandardScaler()))
        steps.append(("model", est))
        return Pipeline(steps)

    return {
        "Linear": (pipe(LinearRegression()), {}),
        "Polynomial (deg 2)": (pipe(LinearRegression(), poly_degree=2), {}),
        "Polynomial (deg 3) + Ridge": (
            pipe(Ridge(), poly_degree=3), {"model__alpha": [10, 100, 1000, 3000]}),
        "Ridge": (pipe(Ridge()), {"model__alpha": np.logspace(-3, 2, 6)}),
        "Lasso": (pipe(Lasso(max_iter=20000)), {"model__alpha": np.logspace(-4, 0, 6)}),
        "ElasticNet": (
            pipe(ElasticNet(max_iter=20000)),
            {"model__alpha": np.logspace(-4, 0, 5), "model__l1_ratio": [0.1, 0.5, 0.9]}),
        "Poly(2) + ElasticNet": (
            pipe(ElasticNet(max_iter=20000), poly_degree=2),
            {"model__alpha": [0.001, 0.01, 0.1], "model__l1_ratio": [0.2, 0.5, 0.8]}),
        "Random Forest": (
            pipe(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), scale=False),
            {"model__n_estimators": [200], "model__max_depth": [None, 8],
             "model__min_samples_leaf": [1, 5]}),
        "Gradient Boosting": (
            pipe(GradientBoostingRegressor(random_state=RANDOM_STATE), scale=False),
            {"model__n_estimators": [100, 300], "model__learning_rate": [0.03, 0.1],
             "model__max_depth": [2, 3]}),
    }


# ----------------------------------------------------------------------------
# Accuracy evaluation
# ----------------------------------------------------------------------------


def within(tol):
    """Share of orders whose predicted time is within +/- tol minutes."""
    return make_scorer(lambda y, p: float(np.mean(np.abs(np.asarray(y) - p) <= tol)))


SCORING = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "MAPE": "neg_mean_absolute_percentage_error",
    "Within5": within(5),
    "Within10": within(10),
}


def tuned(pipeline, grid):
    """Wrap a pipeline in an inner grid search so tuning happens inside each outer fold."""
    if not grid:
        return pipeline
    return GridSearchCV(pipeline, grid, cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                        scoring="neg_root_mean_squared_error", n_jobs=-1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=None)
    ap.add_argument("--out", default=".")
    args, _ = ap.parse_known_args()

    df = load_data(args.csv)
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    print(f"Data: {len(df)} rows | target mean {y.mean():.1f} min, std {y.std():.1f} min\n")

    outer = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
    models = {"Baseline (predict mean)": (DummyRegressor(), {})}
    models.update(get_models(X))

    rows = []
    for name, (pipe, grid) in models.items():
        cv = cross_validate(tuned(pipe, grid), X, y, cv=outer, scoring=SCORING, n_jobs=1)
        r = {"Model": name}
        for k in SCORING:
            vals = cv[f"test_{k}"]
            sign = -1 if k in ("RMSE", "MAE", "MAPE") else 1
            r[k], r[f"{k} sd"] = sign * vals.mean(), vals.std()
        rows.append(r)
        print(f"{name:30s} RMSE {r['RMSE']:.2f} (+/-{r['RMSE sd']:.2f})  MAE {r['MAE']:.2f}  "
              f"R2 {r['R2']:.3f}  within5 {r['Within5']:.0%}  within10 {r['Within10']:.0%}")

    res = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    res.to_csv(f"{args.out}/accuracy_cv_results.csv", index=False)

    show = res.copy()
    show["RMSE"] = show.apply(lambda r: f"{r.RMSE:.2f} +/- {r['RMSE sd']:.2f}", axis=1)
    show["MAE"] = show["MAE"].round(2)
    show["R2"] = show["R2"].round(3)
    show["MAPE"] = (show["MAPE"] * 100).round(1).astype(str) + "%"
    show["Within5"] = (show["Within5"] * 100).round(0).astype(int).astype(str) + "%"
    show["Within10"] = (show["Within10"] * 100).round(0).astype(int).astype(str) + "%"
    print("\n=== Cross-validated accuracy (nested, 5-fold x 2 repeats) ===")
    print(show[["Model", "RMSE", "MAE", "R2", "MAPE", "Within5", "Within10"]].to_string(index=False))

    # ---- Best real model (exclude baseline) ----
    best = res[res.Model != "Baseline (predict mean)"].iloc[0]
    base = res[res.Model == "Baseline (predict mean)"].iloc[0]
    print(f"\nBest model: {best.Model}")
    print(f"  Typical error (MAE): {best.MAE:.1f} min  vs {base.MAE:.1f} min for the naive baseline "
          f"({1 - best.MAE / base.MAE:.0%} lower)")
    print(f"  {best.Within5:.0%} of orders predicted within 5 min, {best.Within10:.0%} within 10 min")

    # ---- Out-of-fold predictions for diagnostics ----
    pipe, grid = models[best.Model]
    oof = cross_val_predict(tuned(pipe, grid), X, y,
                            cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
    diag = X.copy()
    diag["actual"], diag["pred"] = y.values, oof
    diag["resid"] = diag.actual - diag.pred          # + means we under-predicted
    diag["abs_err"] = diag.resid.abs()

    segs = []
    diag["Distance band"] = pd.qcut(diag["Distance_km"], 4, duplicates="drop").astype(str)
    for col in ["Distance band", "Traffic_Level", "Weather", "Time_of_Day", "Vehicle_Type"]:
        g = diag.groupby(col).agg(n=("abs_err", "size"), MAE=("abs_err", "mean"),
                                  bias=("resid", "mean")).round(2).reset_index()
        g.insert(0, "segment", col)
        g = g.rename(columns={col: "group"})
        segs.append(g)
    seg = pd.concat(segs)
    seg.to_csv(f"{args.out}/accuracy_by_segment.csv", index=False)
    print("\n=== Out-of-fold error by segment (bias > 0 = model under-predicts) ===")
    print(seg.to_string(index=False))

    # ---- Plots ----
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].scatter(diag.actual, diag.pred, s=10, alpha=.45)
    lim = [diag.actual.min(), diag.actual.max()]
    ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("Actual (min)"); ax[0].set_ylabel("Predicted (min)")
    ax[0].set_title(f"{best.Model}: out-of-fold predicted vs actual")
    ax[1].scatter(diag.pred, diag.resid, s=10, alpha=.45)
    ax[1].axhline(0, color="r", ls="--"); ax[1].set_xlabel("Predicted (min)")
    ax[1].set_ylabel("Residual (actual - predicted)"); ax[1].set_title("Residuals vs predicted")
    plt.tight_layout(); plt.savefig(f"{args.out}/accuracy_report.png", dpi=130); plt.close()

    print("\nSaved: accuracy_cv_results.csv, accuracy_by_segment.csv, accuracy_report.png")


if __name__ == "__main__":
    main()

Loaded /kaggle/input/datasets/denkuznetz/food-delivery-time-prediction/Food_Delivery_Times.csv
Data: 1000 rows | target mean 56.7 min, std 22.1 min

Baseline (predict mean)        RMSE 22.05 (+/-0.78)  MAE 17.80  R2 -0.002  within5 15%  within10 32%
Linear                         RMSE 10.55 (+/-1.36)  MAE 6.61  R2 0.770  within5 56%  within10 82%
Polynomial (deg 2)             RMSE 11.20 (+/-1.33)  MAE 7.34  R2 0.741  within5 50%  within10 77%
Polynomial (deg 3) + Ridge     RMSE 11.30 (+/-1.34)  MAE 7.56  R2 0.736  within5 47%  within10 77%
Ridge                          RMSE 10.55 (+/-1.36)  MAE 6.62  R2 0.770  within5 55%  within10 82%
Lasso                          RMSE 10.54 (+/-1.35)  MAE 6.60  R2 0.770  within5 56%  within10 82%
ElasticNet                     RMSE 10.55 (+/-1.35)  MAE 6.62  R2 0.770  within5 55%  within10 82%


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.536e+02, tolerance: 2.644e+01
  model = cd_fast.enet_coordinate_descent(


Poly(2) + ElasticNet           RMSE 10.97 (+/-1.32)  MAE 7.12  R2 0.752  within5 52%  within10 78%
Random Forest                  RMSE 11.36 (+/-1.19)  MAE 7.47  R2 0.734  within5 49%  within10 77%
Gradient Boosting              RMSE 11.01 (+/-1.31)  MAE 7.11  R2 0.750  within5 51%  within10 78%

=== Cross-validated accuracy (nested, 5-fold x 2 repeats) ===
                     Model           RMSE   MAE     R2  MAPE Within5 Within10
                     Lasso 10.54 +/- 1.35  6.60  0.770 11.1%     56%      82%
                ElasticNet 10.55 +/- 1.35  6.62  0.770 11.2%     55%      82%
                     Ridge 10.55 +/- 1.36  6.62  0.770 11.2%     55%      82%
                    Linear 10.55 +/- 1.36  6.61  0.770 11.1%     56%      82%
      Poly(2) + ElasticNet 10.97 +/- 1.32  7.12  0.752 12.2%     52%      78%
         Gradient Boosting 11.01 +/- 1.31  7.11  0.750 12.4%     51%      78%
        Polynomial (deg 2) 11.20 +/- 1.33  7.34  0.741 12.6%     50%      77%
Polynomial (deg 